[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/01_read_a_mind.ipynb)

# 01 · Read a Mind

### HAAISS workshop — core notebook 1 of 4

We take an ordinary open model (**Qwen 2.5 7B**), let it answer a question, and then — instead of reading its *words* — capture the **activation vector** inside layer 20 and ask a verbalizer to **describe that vector in English**.

The verbalizer is the AV half of an **NLA** (Natural Language Autoencoder): a small LoRA adapter on the same Qwen base. The other half, introduced in notebook 03, reconstructs an activation from the English caption.

**Setup:** `Runtime → Change runtime type → T4 GPU`, then run every cell top to bottom. First run downloads ~5 GB and takes a few minutes.

## Install

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Configure

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-universal-av-grpo"  # universal verbalizer: ONE adapter, all 28 layers (GRPO-refined)
LAYER      = 20   # default readout layer — the universal adapter serves ALL of 0..27; try others
DEPTH_PCT  = 71   # nearest trained depth tag for layer 20; a conditioning input
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

/home/anicka/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the model (once)

We load Qwen in 4-bit and attach the **AV adapter** (`av` = *activation → verbalization*). The base model is frozen; the adapter is 80 MB.

In [3]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:08<49:49,  8.84s/it]

Loading weights:   1%|          | 2/339 [00:17<49:20,  8.79s/it]

Loading weights:   1%|          | 4/339 [00:18<20:16,  3.63s/it]

Loading weights:   1%|▏         | 5/339 [00:19<15:49,  2.84s/it]

...

Loading weights:  97%|█████████▋| 330/339 [01:37<00:03,  2.80it/s]

Loading weights:  99%|█████████▊| 334/339 [01:38<00:01,  4.54it/s]

Loading weights:  99%|█████████▉| 336/339 [01:38<00:00,  5.33it/s]

Loading weights: 100%|██████████| 339/339 [01:38<00:00,  3.45it/s]

loaded — base + AV adapter on cuda:0


## The two moves
`read_activation(prompt)` runs the model and grabs the layer-20 vector.
`describe(vector)` injects that vector into the verbalizer and reads out a caption.

In [4]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Grab the clean base-model residual at the last prompt token."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    try:
        with model.disable_adapter(), torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
    finally:
        handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to, **gen_kw):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    chat = tok.apply_chat_template([{"role": "user", "content": av_prompt(depth)}],
                                   tokenize=False, add_generation_prompt=True)
    ids = tok.encode(chat, add_special_tokens=False)  # match training: chat-wrapped, no BOS
    pos = ids.index(inject_id)
    input_ids = torch.tensor([ids], device=device)
    emb = model.get_input_embeddings()(input_ids).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    gen_args = dict(do_sample=False)
    gen_args.update(gen_kw)                    # e.g. do_sample=True, temperature=0.8
    attn = torch.ones((1, len(ids)), device=device, dtype=torch.long)
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, inputs_embeds=emb, attention_mask=attn,
                             max_new_tokens=max_new_tokens,
                             pad_token_id=tok.eos_token_id, **gen_args)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

## Try it

Change the prompt. The **OUTPUT** is what Qwen would normally say. The **LAYER-20 READOUT** is what the NLA sees happening *inside* the model at 71% depth — often the topic/structure it has committed to, before the words come out.

In [5]:
# === CHANGE THIS ===
prompt = "Explain how a hash map handles collisions."

activation, reply = read_activation(prompt)
readout = describe(activation)

print("PROMPT :", prompt)
print("\nOUTPUT (what Qwen says):\n ", reply[:400])
print("\nLAYER-20 READOUT (what the NLA sees inside):\n ", readout)

/home/anicka/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


PROMPT : Explain how a hash map handles collisions.

OUTPUT (what Qwen says):
  A hash map, also known as a hash table, is a data structure that implements an associative array abstract data type, a structure that can map keys to values. The primary advantage of a hash map is its ability to provide fast access to elements based on their keys. However, when two different keys hash to the same index in the underlying array (a situation known as a collision), special handling me

LAYER-20 READOUT (what the NLA sees inside):
  - Hash table: data structure for key-value storage
- "how does hash table works" frame: mechanism-explanation response
- "in terms of hashing algorithm" constraint: technical precision (hashing function, collision resolution)
- "in computer science" context: formal, neutral register
- Response strategy: step-by-step procedural explanation (hashing, storing, retrieving keys and values) with a focus on collision handling (chaining or probing)


## Now make it interesting

Try prompts where the *inside* and the *outside* might differ:

- `"Do you have feelings?"` — does the readout mention self-reference / refusal framing before the model hedges out loud?

- `"Translate 'good morning' into French."` — does layer 20 already hold *French* / *translation task*?

- A half sentence: `"The capital of Australia is"` — the answer is committed inside long before the token appears.

Run several and eyeball whether the caption tracks the *content* or just the *surface form*. (This eyeball test is the real evaluation — numbers hide template hallucination.)

In [6]:
for p in ["Do you have feelings?",
          "The capital of Australia is",
          "Write a haiku about rain."]:
    act, rep = read_activation(p)
    print("::", p)
    print("   readout:", describe(act))
    print()

:: Do you have feelings?


   readout: - Self-referential query: "Do you have feelings like humans do?"
- Response strategy: first-person hedged ("I", "As") followed by a contrastive structure ("feelings" vs. "like humans")
- Competing interpretations: literal emotional experience vs. metaphorical attribution of human-like qualities
- Tone: philosophical and introspective, with a mild tension between direct yes/no and nuanced explanatory framing
- Response register: introspective and self-aware, acknowledging the question as a fundamental one about consciousness and agency.



:: The capital of Australia is


   readout: - Capital city of Australia: Canberra
- Geographic and geopolitical knowledge retrieval: "The capital of Australia" as factual location query
- Direct declarative answer: "Canberra" as primary response token
- Contextual constraint: "What is the capital of Australia?" as simple factual question
- Response strategy: short, confident, direct answer without hedging or elaboration
- Tension between "capital" and "Australia": model must resolve to Canberra, suppressing alternative interpretations like Sydney or Melbourne.



:: Write a haiku about rain.


   readout: - Poetic generation task: "Write a short poem starting with rain falls vertically down"
- Imagery and sensory tokens ("wet asphalt", "cherry blossom") as constraints
- Line-break and meter patterns favored over prose
- Tension between literal verticality and metaphorical stillness, with "poem" and "spring evening" favoring metaphorical stillness
- Response strategy: opening line with "rain falls" followed by a volta or stasis, using "cherry blossoms" as a visual anchor
- Tone: lyrical and evocative, with a



## Where do confabulated entities come from?

If the hash-map readout above named a programming language the prompt never
mentioned (C# is a favorite), you caught **entity slot-filling**. The training
corpus and descriptions contain no occurrence of "hash" or "C#"; the detail
came from the decoder rather than a memorized training example. That does not
tell us whether the source activation lacked the entity or the AV failed to
recover it.

Two experiments that separate *read* from *filled-in*:

1. **Pin the entity in the prompt.** If it transfers, that is positive evidence
   for recovery. If it does not, the test exposes readout loss; it does not prove
   the base activation lacked the entity.
2. **Re-sample the readout.** Stable coarse framing is decoded more robustly than
   details that change across samples. Stability is evidence, not proof of origin.

In [7]:
# 1) entity pinned in the prompt — test whether the AV recovers it
act_java, _ = read_activation("Explain how a Java HashMap handles collisions.")
print("JAVA-PINNED READOUT:\n ", describe(act_java), "\n")

# 2) entity unpinned — re-sample and watch which parts move
act, _ = read_activation("Explain how a hash map handles collisions.")
for t in range(3):
    print(f"sample {t}:\n ", describe(act, do_sample=True, temperature=0.8), "\n")

JAVA-PINNED READOUT:
  - Technical query about C++ programming: "How does the compiler implement" and "hash code" (likely "hashcode") are key concepts.
- Response strategy: direct explanatory, focusing on the `std::hash` function or hash-based data structures like unordered containers.
- Tone: neutral technical register, expecting a step-by-step answer.
- Contextual constraint: "In C++" narrows the response to C++ standard library mechanisms.
- Tension between "hash code" and "hashing mechanism": the former may refer to the `std::hash` function, while the latter 



sample 0:
  - Technical CS query: "how does a red-black tree handle node coloring when inserting a new node"
- Response strategy: step-by-step algorithmic description (insertion procedure)
- Key concepts: node color (black/white), violation of properties (RB-tree invariant), recoloring and rotation operations
- Tension between "node coloring" (color property assignment) and "handle" (rebalancing steps)
- Context: "how does it work internally" favors detailed procedural answer over high-level overview
- Output structure: sequence of recolorings followed by rotations to restore the RB 



sample 1:
  - Technical CS query: "How does a hash table works in a hash code algorithm?"
- Focus on "how does" and "works"
- Response strategy: mechanistic, step-by-step explanation of hash-table lookup process
- Key concept: hash function mapping keys to indices (buckets)
- Secondary thread: collision resolution (hash collisions, chaining, probing)
- Tone: neutral explanatory register, formal but accessible prose
- Output structure: likely beginning with "In a hash table, the hash code algorithm assigns" or "When a key is inserted into a hash table," followed by a description 



sample 2:
  - Hash table lookup: technical CS concept (hashing algorithm, unordered_map) active
- "how does a hash table works" parsed as interrogative structure ("How does X works?")
- Response strategy: direct explanatory answer ("In a hash table,") with mechanism focus on "collision resolution"
- Contextual constraint: "during hashing process" keeps focus on internal operation rather than external application
- Tone: neutral, instructional register, expecting a clear, step-by-step explanation of collision-handling strategies like chaining or open addressing. 



## Brain in the jar — the whole depth ladder

So far we read ONE depth. The universal adapter was trained across all 28
Qwen layers, conditioned on thirteen rounded depth tags,
and the real experiment is reading the *same moment of thought* at every
depth and watching where the truth appears.

From our test run, so you know what you're looking at: it is NOT a smooth
"tokens → situation → plan" climb. The shallow rungs confabulate whole
scenes — the verbalizer meets an activation carrying only vague features
and its decoder prior invents a complete, coherent, WRONG situation (a
speed-of-light question at 4%, a purple-banana weight argument at 17%).
The prompt's actual content ("recipe needs eggs, she's short") first
flickers around 32% and locks in near 63%. But watch the deep rungs: they
stay on-topic and start hallucinating *details* nobody asked for
(gluten-free, no yogurt, cereal milk). Confabulation doesn't vanish with
depth — it changes grain, from whole-scene to detail.

One forward captures all depths at once (hooks are free); each rung then
costs one `describe()` (~10 s on a T4). We read the activations under
`disable_adapter()` — the NLA was trained on clean base activations, so
that's the distribution it expects. Seven rungs below; switch to
`DEPTH_LAYERS_FULL` for all thirteen.

In [8]:
# trained depth grid (pct of 28 layers -> block index), from the repo's nla_lib
DEPTH_LAYERS_FULL = {4: 1, 10: 3, 17: 5, 25: 7, 32: 9, 40: 11, 47: 13,
                     55: 15, 63: 18, 71: 20, 80: 22, 90: 25, 96: 27}
DEPTH_LAYERS = {p: DEPTH_LAYERS_FULL[p] for p in (4, 17, 32, 47, 63, 80, 96)}

prompt = "The recipe calls for two eggs, but she only has one, so"

chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                               tokenize=False, add_generation_prompt=True)
inp = tok(chat, return_tensors="pt").to(device)

grabs, handles = {}, []
def _mk(pct):
    def hook(mod, i, o):
        h = o[0] if isinstance(o, tuple) else o
        if pct not in grabs:
            grabs[pct] = h[:, -1, :].detach().squeeze(0)
    return hook
for pct, L in DEPTH_LAYERS.items():
    handles.append(get_layers(model)[L].register_forward_hook(_mk(pct)))
try:
    # disable_adapter: the NLA was trained on CLEAN base activations, so read
    # the base residual, not base+AV-LoRA (the adapter is ON by default)
    with model.disable_adapter(), torch.no_grad():
        model(**inp)                # ONE forward pass captures every rung
finally:
    for h_ in handles:
        h_.remove()

for pct in sorted(grabs):
    print(f"\n{'='*22} depth {pct}% (layer {DEPTH_LAYERS[pct]}) {'='*22}")
    print(describe(grabs[pct], depth=pct, max_new_tokens=90))


====================== depth 4% (layer 1) ======================


-
- Interrogative frame: "How fast" as a rate-of-change query
- Temporal and spatial co-indexing: "two objects" and "move toward each other"
- Numerical anchor: "10 meters per second"
- Spatial relationship: "meet at a point in between" (converging motion)
- Syntactic structure: "The speed of one object is 8 m/s" (parallel clause)

====================== depth 17% (layer 5) ======================


-
- Conditional frame: "If a blue whale is drinking and the ocean is calm" with subject ("a blue whale"), verb ("is drinking"), and locative ("the ocean")
- Syntactic role: "and" as a conjunction linking two simultaneous states
- Semantic cluster: "blue whale" and "ocean" in aquatic biology and marine ecology
- Question structure: "what color is the water" as a direct interrogative

====================== depth 32% (layer 9) ======================


-
- Syntactic frame: "you said you were going to [infinitive]" with "but" as a contrastive conjunction
- Coreference chain: "it" -> "the juice" -> "spill"
- Temporal dependency: "and now" linking past action to present state
- Predicted completion: "and now what?" with a question-answering response strategy
- Competing interpretations: "and

====================== depth 47% (layer 13) ======================


-
- Cooking-instructional frame: "and then the water boils" and "the pot" as subject
- Temporal dependency: "you check the pot again" with "the water has just boiled"
- Next-token predictions: procedural continuations ("then you", "next step", "before")
- Competing interpretations: "so you can add the pasta" (action initiation vs. conditional clause)
- Response strategy: continuation of the

====================== depth 63% (layer 18) ======================


-
- Cooking recipe context: "if you're short on eggs" and "you could use" as a substitution frame
- Quantitative constraint: "one egg short" activating a missing-ingredient problem
- Response strategy: substitution (e.g., milk, yogurt, or a leavening agent) rather than procedural continuation
- Tension between practical advice and creative workaround due to "what would you do?" framing
- Competing interpretations

====================== depth 80% (layer 22) ======================


-
- Cooking-problem resolution: "what can i do instead of" and "gluten-free oat flour" as a substitution constraint
- Response strategy: "you could use" or "try using" followed by a direct replacement suggestion (e.g., "regular flour")
- Contextual constraints: "gluten-free" and "no store nearby" narrowing the solution space toward readily available pantry ingredients like all-purpose flour or almond meal
-

====================== depth 96% (layer 27) ======================


-
- Culinary-procedural frame: "recipe calls for milk but I have none" with "substitute" as the primary action verb
- Contextual constraint: "what should I use instead?" as a direct interrogative seeking a single replacement ingredient
- Response strategy: "creamer or heavy cream" as likely substitutes for milk in a recipe context
- Tension between "creamers" (plant-based alternatives) and "heavy


**The game: at which rung does the prompt first appear, and where does
it lock?** In our run eggs flickered at 32%, the egg-shortage/substitution
situation locked at 63%, and 80-96% stayed on-topic but bolted on
invented constraints (a gluten-free cookie, missing yogurt). Your exact
rungs will shift — this is a bf16 anchor and Colab runs 4-bit — so read
the shape, not the layer numbers.

Three things one ladder shows:

1. **A fluent readout is not a faithful one.** The shallow scenes were
   perfectly coherent and entirely invented — same lesson as the entity
   demo above, now as a function of depth.
2. **Verbalizable content concentrates in a band** (~47-80% here), the
   same band where notebook 05's Jacobian lens ignites. Two different
   instruments, one boundary.
3. **Confabulation changes grain, it doesn't disappear.** Shallow rungs
   invent the whole scene; deep rungs get the scene right and invent the
   specifics. Reading the gaps honestly — knowing WHICH part of a fluent
   readout to trust at which depth — is what separates an interpretability
   tool from a storyteller. Current NLAs are still part storyteller, and
   now you can see exactly where.

---
### ✅ Self-check
Verified Colab anchors:
- the model loads without OOM on a **free T4** (4-bit uses ~5–6 GB),
- for `"Explain how a hash map..."` the readout should usually preserve the technical gist. Repeated unrelated output suggests an interface problem such as the injection scale; wrong details can also be ordinary AV confabulation.

In [9]:
act, _ = read_activation("Explain how a hash map handles collisions.")
out = describe(act)
print(out)
assert len(out) > 10, "empty readout — check the injection token / adapter load"
print("\nself-check: readout non-empty ✓  (now eyeball that it is on-topic)")

- Hash table: data structure for key-value storage
- "how does hash table works" frame: mechanism-explanation response
- "in terms of hashing algorithm" constraint: technical precision (hashing function, collision resolution)
- "in computer science" context: formal, neutral register
- Response strategy: step-by-step procedural explanation (hashing, storing, retrieving keys and values) with a focus on collision handling (chaining or probing)

self-check: readout non-empty ✓  (now eyeball that it is on-topic)
